# Notebook 06 — TargetDiff Generation: EGFR T790M Mutant (4I22)

Day 8 goals:
- Mirror notebook 04, use 4I22 pocket instead
- Generate 1000+ molecules
- Save in batches for crash recovery


In [ ]:
import os, sys, glob
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'external', 'targetdiff'))

POCKET_FILE = f'{PROJECT_ROOT}/data/pockets/4I22_pocket.pdb'
OUTPUT_SDF  = f'{PROJECT_ROOT}/results/generated/4I22_raw.sdf'
CHECKPOINT  = f'{PROJECT_ROOT}/external/targetdiff/checkpoints/pretrained_diffusion.pt'
SAMPLE_SCRIPT = f'{PROJECT_ROOT}/external/targetdiff/scripts/sample_for_pocket.py'

os.makedirs(f'{PROJECT_ROOT}/results/generated', exist_ok=True)

import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
# ── Run TargetDiff for T790M ────────────────────────────────────────────────
import subprocess

NUM_SAMPLES = 100
BATCH_ID    = 0   # increment per batch

batch_out = f'{PROJECT_ROOT}/results/generated/4I22_batch{BATCH_ID:02d}.sdf'
cmd = [
    'python', SAMPLE_SCRIPT,
    '--pdb_file', POCKET_FILE,
    '--num_samples', str(NUM_SAMPLES),
    '--out_sdf', batch_out,
    '--checkpoint', CHECKPOINT,
]
print('Command ready. Uncomment to execute:')
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout[-2000:])

In [ ]:
# ── Merge batches ────────────────────────────────────────────────────────────
from rdkit import Chem

batch_files = sorted(glob.glob(f'{PROJECT_ROOT}/results/generated/4I22_batch*.sdf'))
writer = Chem.SDWriter(OUTPUT_SDF)
total = 0
for bf in batch_files:
    for mol in Chem.SDMolSupplier(bf, removeHs=False):
        if mol:
            writer.write(mol)
            total += 1
writer.close()
print(f'Merged {total} T790M molecules → {OUTPUT_SDF}')